# SDSS-like Mock-Training Reference Run

This notebook records the reference mock generation, training, evaluation, uncertainty calibration, and GH06 external-validation workflow. It uses the native SDSS instrumental-resolution channel, mixed independent/covariant AGN priors, line-broadening augmentation, and an empirical validation-risk model.

## 0. Runtime Notes

- Use the `dl-veldisp` conda environment as the Jupyter kernel.
- Set `SPS_HOME` before starting Jupyter.
- The 220,000-spectrum reference run requires substantial disk space and compute time.
- Set `FORCE_REGENERATE` or `FORCE_RETRAIN` to `True` only when an existing product should be replaced.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'scripts').is_dir():
    raise RuntimeError('Start Jupyter from the repository root or its notebooks directory.')
SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
os.chdir(PROJECT_ROOT)

if not os.environ.get('SPS_HOME'):
    raise RuntimeError('Set SPS_HOME to the FSPS installation before running this notebook.')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib-cache'))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

env = os.environ.copy()

def shell_join(cmd):
    return ' '.join(shlex.quote(str(part)) for part in cmd)

def run_command(cmd, *, skip_if_exists=None, force=False):
    if skip_if_exists is not None:
        skip_path = Path(skip_if_exists)
        if skip_path.exists() and not force:
            print(f'Skipping because {skip_path} already exists. Set force=True to rerun.')
            return
    print(shell_join(cmd))
    t0 = time.time()
    subprocess.run([str(part) for part in cmd], check=True, env=env)
    print(f'Finished in {(time.time() - t0) / 60:.1f} min')

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)
print('SPS_HOME:', os.environ.get('SPS_HOME'))

## 1. Global Run Configuration

This cell defines the mock parameter space, AGN covariance prior, training schedule, and output names for the 220,000-spectrum reference run.

The reference configuration includes:

- native SDSS instrumental resolution as a model input;
- line-shape branches for CaT, Fe/metal, and Ca H+K;
- controlled line-broadening augmentation;
- an old/intermediate-bulge stellar-population prior;
- broad sampling from 30 to 700 km/s;
- mixed independent and EV1/Baldwin-correlated AGN parameters; and
- validation-based uncertainty and local quality-risk calibration.

In [ ]:
MOCK_RUN_TAG = 'sdss_reference_native_resolution'
RUN_TAG = 'sdss_reference_lineshape_aug'

N_SAMPLES = 220_000
SAMPLE_LABEL = f'{N_SAMPLES // 1000}k'
SEED = 20260630
FORCE_REGENERATE = False
FORCE_RETRAIN = False  # RUN_TAG points to a new checkpoint, so training will not be skipped.

WAVE_MIN = 1200
WAVE_MAX = 9000
N_PIXELS = 6000

N_Z = 12
N_AGE = 24
LOGZSOL_MIN = -2.0
LOGZSOL_MAX = 0.5
AGE_MIN_GYR = 0.01
AGE_MAX_GYR = 13.5

SFH_OLD_BULGE_PROB = 0.62
SFH_INTERMEDIATE_BULGE_PROB = 0.28
SFH_YOUNG_BURST_PROB = 0.25
SFH_YOUNG_BURST_MIN = 0.005
SFH_YOUNG_BURST_MAX = 0.08
SFH_RARE_YOUNG_BURST_PROB = 0.05
SFH_RARE_YOUNG_BURST_MAX = 0.15

SIGMA_MIN = 30
SIGMA_MAX = 700
SIGMA_SAMPLING_MODE = 'science_bins'
SCIENCE_SIGMA_BIN_EDGES = '30,60,80,120,180,250,400,700'
SCIENCE_SIGMA_BIN_PROBS = '0.07,0.08,0.18,0.20,0.19,0.16,0.12'
SCIENCE_SIGMA_BIN_LOSS_WEIGHTS = '1.4,1.5,1.9,1.9,1.8,1.6,1.4'
HOST_FRACTION_BIN_PROBS = '0.20,0.38,0.42'

# Approximate empirical AGN covariance prior: EV1 ties FeII/[O III]/H beta,
# and log L5100 introduces a Baldwin-like EW trend. The code still supports
# AGN_PARAMETER_MODE = 'independent' for an uncorrelated nuisance prior.
AGN_PARAMETER_MODE = 'mixed_ev1_baldwin'
AGN_COVARIANT_FRACTION = 0.45
AGN_LOG_L5100_MIN = 42.0
AGN_LOG_L5100_MAX = 46.8

REDSHIFT_MIN = 0.01
REDSHIFT_MAX = 0.90
REDSHIFT_SAMPLING_MODE = 'science_bins'
REDSHIFT_BIN_EDGES = '0.01,0.06,0.15,0.35,0.90'
REDSHIFT_BIN_PROBS = '0.35,0.25,0.25,0.15'
SNR_MIN = 5
SNR_MAX = 80

SDSS_SIGMA_INST = 70.0
SDSS_SIGMA_INST_MIN = 55.0
SDSS_SIGMA_INST_MAX = 95.0
TEMPLATE_SIGMA_INST = 0.0
COMMON_RESOLUTION_SIGMA = 0.0
SKY_RESIDUAL_PROB = 0.60
SKY_RESIDUAL_SCALE = 1.35
TELLURIC_RESIDUAL_PROB = 0.40

TRAIN_FRAC = 0.80
VAL_FRAC = 0.10
TEST_FRAC = 0.10

TRAIN_EPOCHS = 34
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 256
NUM_WORKERS = 4
VAL_NUM_WORKERS = 0  # keep validation stable in VS Code/Jupyter on macOS
VAL_MAX_BATCHES = 32  # fast validation batches on non-checkpoint epochs
FULL_VAL_EVERY = 4  # full validation cadence for checkpoint selection
LR = 6e-4
EARLY_STOPPING_PATIENCE = 7
EARLY_STOPPING_MIN_DELTA = 0.25
MIN_FULL_TRAINING_EPOCHS = 16

TRAIN_CHANNELS = 'flux,ivar,mask,sigma_flux,inst_sigma,region_cat,region_fe,region_cahk,agn_log_l5100,sigma_mask'
LOW_UNCERTAINTY_PERCENTILE = 35.0
QUALITY_MIN_HOST = 0.50
QUALITY_MIN_SIGMA_MASK_FRACTION = 0.015
QUALITY_MIN_VALID_FRACTION = 0.20
QUALITY_MIN_RESOLUTION_RATIO = 1.00
QUALITY_HIGH_CONF_MIN_RESOLUTION_RATIO = 1.25
QUALITY_MAX_FIT_RECON_MAE = 0.18
QUALITY_MAX_SIGMA_RECON_MAE = 0.15
QUALITY_MAX_SIGMA_RESIDUAL_TO_FEATURE = 0.85
QUALITY_MIN_HOST_SIGMA_FRACTION = 0.42
QUALITY_MIN_ABSORPTION_DEPTH = 0.025
REAL_SDSS_SYSTEMATIC_FRAC = 0.12
REAL_SDSS_LOWRES_EXTRA_FRAC = 0.25

QUALITY_RISK_CALIBRATION_MAX_SAMPLES = 7000
QUALITY_RISK_NEIGHBORS = 112
QUALITY_RISK_GOLD_MAX_EXPECTED_ERROR = 30.0
QUALITY_RISK_SILVER_MAX_EXPECTED_ERROR = 45.0

MOCK_DIR = PROJECT_ROOT / 'mock_data'
MODEL_DIR = PROJECT_ROOT / 'models'
FIGURE_DIR = PROJECT_ROOT / 'figures'
MOCK_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

H5_PATH = MOCK_DIR / f'mock_{MOCK_RUN_TAG}_{SAMPLE_LABEL}.h5'
MODEL_PATH = MODEL_DIR / f'best_{RUN_TAG}.pth'
TRAIN_HISTORY_JSON = MODEL_DIR / f'{RUN_TAG}_training_history.json'
EVAL_PDF = FIGURE_DIR / f'{RUN_TAG}_test_evaluation.pdf'
EVAL_JSON = FIGURE_DIR / f'{RUN_TAG}_test_metrics.json'
CALIBRATION_JSON = FIGURE_DIR / f'{RUN_TAG}_calibration.json'
COVERAGE_FIG = FIGURE_DIR / f'{RUN_TAG}_parameter_coverage.pdf'
MOCK_EXAMPLE_DIR = FIGURE_DIR / f'{RUN_TAG}_mock_examples'
GREENE_EXTERNAL_CSV = FIGURE_DIR / f'{RUN_TAG}_greene2006_external_validation.csv'
GREENE_EXTERNAL_PDF = FIGURE_DIR / f'{RUN_TAG}_greene2006_external_validation.pdf'
GREENE_RECON_PDF = FIGURE_DIR / f'{RUN_TAG}_greene2006_reconstruction_diagnostics.pdf'
GREENE_RECON_CSV = FIGURE_DIR / f'{RUN_TAG}_greene2006_reconstruction_diagnostics.csv'

SDSS_REFERENCE_DIR = PROJECT_ROOT / 'sdss_reference_spectra' / 'greene2006'
GREENE_MANIFEST = SDSS_REFERENCE_DIR / 'greene2006_sdss_manifest.csv'
GREENE_LITERATURE_SIGMA = SDSS_REFERENCE_DIR / 'greene2006_literature_sigma.csv'
GREENE_LITERATURE_TEMPLATE = SDSS_REFERENCE_DIR / 'greene2006_literature_sigma_template.csv'

# Backward-compatible names used by the external-validation cells.
GREENE_MANIFEST_CSV = GREENE_MANIFEST
GREENE_LITERATURE_SIGMA_CSV = GREENE_LITERATURE_SIGMA
GREENE_LITERATURE_TEMPLATE_CSV = GREENE_LITERATURE_TEMPLATE

EXTRA_FEII_TEMPLATE_DIRS = [
    PROJECT_ROOT / 'scripts' / 'fe_template',
    PROJECT_ROOT / 'reference' / 'feii_templates',
]
EXTRA_FEII_TEMPLATES = sorted(
    {
        str(path)
        for directory in EXTRA_FEII_TEMPLATE_DIRS
        if directory.exists()
        for path in directory.glob('*.txt')
    }
)
print('Extra FeII templates:', len(EXTRA_FEII_TEMPLATES))

print('Mock run tag:', MOCK_RUN_TAG)
print('Training/output run tag:', RUN_TAG)
print('N samples:', N_SAMPLES)
print('HDF5 output:', H5_PATH)
print('Model output:', MODEL_PATH)


## 2. Check Device

Confirm that the Jupyter kernel can access Apple MPS. The training script selects `mps`, then `cuda`, then `cpu` according to availability.

In [ ]:
import torch

print('torch:', torch.__version__)
print('mps built:', torch.backends.mps.is_built())
print('mps available:', torch.backends.mps.is_available())
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print('selected device:', device)

## 3. Generate Large SDSS-like Mock Dataset

Run `scripts/generate_sdss_quasar_mocks.py`. The HDF5 output contains fixed data splits, flux-precision and mask arrays, host and AGN components, and the physical and nuisance labels.

In [ ]:
generate_cmd = [
    sys.executable,
    SCRIPTS_DIR / 'generate_sdss_quasar_mocks.py',
    '--output', H5_PATH,
    '--n-samples', N_SAMPLES,
    '--seed', SEED,
    '--fsps-home', os.environ['SPS_HOME'],
    '--wave-min', WAVE_MIN,
    '--wave-max', WAVE_MAX,
    '--n-pixels', N_PIXELS,
    '--n-z', N_Z,
    '--n-age', N_AGE,
    '--logzsol-min', LOGZSOL_MIN,
    '--logzsol-max', LOGZSOL_MAX,
    '--age-min-gyr', AGE_MIN_GYR,
    '--age-max-gyr', AGE_MAX_GYR,
    '--sfh-old-bulge-prob', SFH_OLD_BULGE_PROB,
    '--sfh-intermediate-bulge-prob', SFH_INTERMEDIATE_BULGE_PROB,
    '--sfh-young-burst-prob', SFH_YOUNG_BURST_PROB,
    '--sfh-young-burst-min', SFH_YOUNG_BURST_MIN,
    '--sfh-young-burst-max', SFH_YOUNG_BURST_MAX,
    '--sfh-rare-young-burst-prob', SFH_RARE_YOUNG_BURST_PROB,
    '--sfh-rare-young-burst-max', SFH_RARE_YOUNG_BURST_MAX,
    '--sigma-min', SIGMA_MIN,
    '--sigma-max', SIGMA_MAX,
    '--sigma-sampling-mode', SIGMA_SAMPLING_MODE,
    '--science-sigma-bin-edges', SCIENCE_SIGMA_BIN_EDGES,
    '--science-sigma-bin-probs', SCIENCE_SIGMA_BIN_PROBS,
    '--host-fraction-bin-probs', HOST_FRACTION_BIN_PROBS,
    '--agn-parameter-mode', AGN_PARAMETER_MODE,
    '--agn-covariant-fraction', AGN_COVARIANT_FRACTION,
    '--agn-log-l5100-min', AGN_LOG_L5100_MIN,
    '--agn-log-l5100-max', AGN_LOG_L5100_MAX,
    '--redshift-sampling-mode', REDSHIFT_SAMPLING_MODE,
    '--redshift-bin-edges', REDSHIFT_BIN_EDGES,
    '--redshift-bin-probs', REDSHIFT_BIN_PROBS,
    '--sdss-sigma-inst', SDSS_SIGMA_INST,
    '--sdss-sigma-inst-min', SDSS_SIGMA_INST_MIN,
    '--sdss-sigma-inst-max', SDSS_SIGMA_INST_MAX,
    '--template-sigma-inst', TEMPLATE_SIGMA_INST,
    '--common-resolution-sigma', COMMON_RESOLUTION_SIGMA,
    '--sky-residual-prob', SKY_RESIDUAL_PROB,
    '--sky-residual-scale', SKY_RESIDUAL_SCALE,
    '--telluric-residual-prob', TELLURIC_RESIDUAL_PROB,
    '--redshift-min', REDSHIFT_MIN,
    '--redshift-max', REDSHIFT_MAX,
    '--snr-min', SNR_MIN,
    '--snr-max', SNR_MAX,
    '--train-frac', TRAIN_FRAC,
    '--val-frac', VAL_FRAC,
    '--test-frac', TEST_FRAC,
    '--sdss-reference-dir', SDSS_REFERENCE_DIR,
]

for template_path in EXTRA_FEII_TEMPLATES:
    generate_cmd.extend(['--feii-template', template_path])


def mock_h5_generation_complete(path, expected_n):
    if not Path(path).exists():
        return False
    try:
        import h5py
        with h5py.File(path, 'r') as hf:
            has_complete_flag = bool(hf.attrs.get('generation_complete', False))
            shape_ok = 'spectra' in hf and int(hf['spectra'].shape[0]) == int(expected_n)
            label_keys_ok = all(key in hf for key in ['agn_log_l5100', 'agn_ev1', 'agn_covariant_draw', 'oiii5007_ew', 'fevii5721_ew'])
            return has_complete_flag and shape_ok and label_keys_ok
    except Exception as exc:
        print(f'Mock HDF5 completeness check failed for {path}: {exc}')
        return False

if H5_PATH.exists() and not mock_h5_generation_complete(H5_PATH, N_SAMPLES):
    print(f'Removing incomplete mock HDF5 before regeneration: {H5_PATH}')
    H5_PATH.unlink()

run_command(generate_cmd, skip_if_exists=H5_PATH, force=FORCE_REGENERATE)


## 4. Inspect Parameter-Space Coverage

Inspect the data splits and label distributions before training. This section also writes a parameter-coverage PDF.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

label_keys = [
    'sigma_star',
    'host_fraction',
    'snr',
    'redshift',
    'mass_weighted_age_gyr',
    'mass_weighted_logzsol',
    'young_fraction',
    'agn_log_l5100',
    'agn_ev1',
    'r_feii',
    'feii_fwhm',
    'balmer_fwhm',
    'hbeta_ew',
    'oiii5007_ew',
    'sigma_inst_kms',
    'sigma_resolution_ratio',
    'cat_contamination_ew',
    'telluric_residual_strength',
    'sky_residual_strength',
    'cat_mask_fraction',
    'fe_mask_fraction',
    'ca_hk_mask_fraction',
]

with h5py.File(H5_PATH, 'r') as hf:
    n = len(hf['spectra'])
    print('n spectra:', n)
    print('spectra shape:', hf['spectra'].shape)
    print('wave range:', float(hf['wave_grid'][0]), float(hf['wave_grid'][-1]))
    print('split counts:', hf.attrs.get('split_counts_json', 'missing'))
    print('FeII templates:', hf.attrs.get('feii_template_manifest_json', 'missing')[:1000])
    print('SDSS references:', hf.attrs.get('sdss_noise_reference_manifest_json', 'missing')[:1000])
    print('\nScalar label summary:')
    label_data = {}
    for key in label_keys:
        values = hf[key][:].astype(float)
        label_data[key] = values
        print(f'{key:24s} min={np.nanmin(values):9.3f} median={np.nanmedian(values):9.3f} max={np.nanmax(values):9.3f}')
    sample_idx = np.linspace(0, n - 1, min(n, 5000), dtype=int)
    valid_fraction = hf['valid_pixels'][sample_idx].mean(axis=1)
    sigma_fraction = hf['sigma_mask'][sample_idx].mean(axis=1)
    print(f'valid pixel fraction sample: median={np.median(valid_fraction):.3f}, min={valid_fraction.min():.3f}, max={valid_fraction.max():.3f}')
    print(f'sigma-window fraction sample: median={np.median(sigma_fraction):.3f}, min={sigma_fraction.min():.3f}, max={sigma_fraction.max():.3f}')

n_cols = 4
n_rows = int(np.ceil(len(label_keys) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 2.6 * n_rows), constrained_layout=True)
for ax, key in zip(np.ravel(axes), label_keys):
    ax.hist(label_data[key], bins=50, color='0.35', alpha=0.82)
    ax.set_title(key, fontsize=9)
    ax.tick_params(labelsize=8)
for ax in np.ravel(axes)[len(label_keys):]:
    ax.set_axis_off()
fig.suptitle(f'{RUN_TAG}: parameter coverage, N={N_SAMPLES}', fontsize=12)
fig.savefig(COVERAGE_FIG)
print('saved:', COVERAGE_FIG)
plt.show()

## 5. Train Hybrid Model

The first three epochs initialize the velocity-dispersion head with host-visible, higher-S/N spectra. Full training then uses every training sample and the complete loss. On Apple MPS, the trainer disables gradient paths that produced non-finite values in correlation-heavy reconstruction terms. Stable interval, pairwise, log-sigma, host, stellar-population, uncertainty, and reconstruction supervision remains active. The learning-rate scheduler begins after warmup.


In [ ]:
train_cmd = [
    sys.executable,
    SCRIPTS_DIR / 'train_hybrid_sdss.py',
    '--h5', H5_PATH,
    '--save', MODEL_PATH,
    '--epochs', TRAIN_EPOCHS,
    '--batch-size', BATCH_SIZE,
    '--num-workers', NUM_WORKERS,
    '--val-num-workers', VAL_NUM_WORKERS,
    '--val-max-batches', VAL_MAX_BATCHES,
    '--full-val-every', FULL_VAL_EVERY,
    '--lr', LR,
    '--channels', TRAIN_CHANNELS,
    '--sigma-warmup-epochs', 3,
    '--sigma-warmup-min-host', 0.30,
    '--sigma-warmup-min-snr', 20.0,
    '--full-loss-ramp-epochs', 1,
    '--extreme-sigma-low', 80.0,
    '--extreme-sigma-high', 500.0,
    '--extreme-sigma-sampler-boost', 1.5,
    '--extreme-sigma-loss-boost', 1.0,
    '--science-sigma-bin-edges', SCIENCE_SIGMA_BIN_EDGES,
    '--science-sigma-bin-loss-weights', SCIENCE_SIGMA_BIN_LOSS_WEIGHTS,
    '--science-bin-sampler-boost', 1.25,
    '--science-bin-loss-boost', 1.10,
    '--sigma-bin-consistency-weight', 4.5,
    '--sigma-pairwise-weight', 1.0,
    '--sigma-pairwise-min-delta', 18.0,
    '--sigma-pairwise-scale', 55.0,
    '--sigma-log-weight', 0.30,
    '--sigma-bin-class-weight', 0.30,
    '--line-broadening-aug-prob', 0.35,
    '--line-broadening-aug-min-kms', 35.0,
    '--line-broadening-aug-max-kms', 150.0,
    '--line-broadening-aug-weight', 3.0,
    '--line-broadening-order-weight', 1.0,
    '--sigma-weight', 20.0,
    '--sigma-distribution-weight', 0.5,
    '--recon-weight', 25.0,
    '--sigma-recon-weight', 8.0,
    '--sigma-deriv-weight', 2.0,
    '--sigma-deriv-corr-weight', 0.0,
    '--host-sigma-fraction-weight', 2.0,
    '--host-weight', 15.0,
    '--smooth-weight', 2.0,
    '--sfh-weight', 0.5,
    '--uncertainty-weight', 0.05,
    '--checkpoint-recon-score-weight', 100.0,
    '--checkpoint-host-score-weight', 50.0,
    '--checkpoint-bin-rmse-weight', 0.70,
    '--checkpoint-lowmid-rmse-weight', 0.45,
    '--quality-loss-min-weight', 0.25,
    '--quality-loss-target-sigma-pixels', 350.0,
    '--quality-loss-host-floor', 0.10,
    '--quality-loss-resolution-min-weight', 0.65,
    '--quality-loss-resolution-full-ratio', 2.5,
    '--early-stopping-patience', EARLY_STOPPING_PATIENCE,
    '--early-stopping-min-delta', EARLY_STOPPING_MIN_DELTA,
    '--min-full-training-epochs', MIN_FULL_TRAINING_EPOCHS,
    '--history-json', TRAIN_HISTORY_JSON,
]

def training_run_complete(history_path, model_path, expected_epochs):
    history_path = Path(history_path)
    model_path = Path(model_path)
    if not model_path.exists() or not history_path.exists():
        return False
    try:
        with open(history_path, 'r', encoding='utf-8') as handle:
            payload = json.load(handle)
        history = payload.get('history', [])
        return bool(payload.get('stopped_early', False)) or len(history) >= int(expected_epochs)
    except Exception as exc:
        print(f'Training completeness check failed: {exc}')
        return False

if FORCE_RETRAIN:
    run_command(train_cmd, force=True)
elif training_run_complete(TRAIN_HISTORY_JSON, MODEL_PATH, TRAIN_EPOCHS):
    print(f'Skipping completed training run: {MODEL_PATH}')
elif MODEL_PATH.exists() and TRAIN_HISTORY_JSON.exists():
    print('Found an incomplete/interrupted training run; resuming from the current best checkpoint.')
    resume_cmd = list(train_cmd) + ['--resume-from', MODEL_PATH]
    run_command(resume_cmd, force=True)
else:
    run_command(train_cmd, force=True)


## 6. Evaluate on Held-Out Test Split

Evaluate the final checkpoint only on the independent test split. The outputs include direct predictions, the validation-derived linear correction for mock diagnostics, and calibrated uncertainty-coverage measurements.

In [ ]:
eval_cmd = [
    sys.executable,
    SCRIPTS_DIR / 'evaluate_hybrid_sdss.py',
    '--h5', H5_PATH,
    '--model', MODEL_PATH,
    '--split', 'test',
    '--calibrate-split', 'val',
    '--output', EVAL_PDF,
    '--metrics-json', EVAL_JSON,
    '--batch-size', EVAL_BATCH_SIZE,
    '--num-workers', NUM_WORKERS,
    '--calibration-json-out', CALIBRATION_JSON,
    '--low-uncertainty-percentile', LOW_UNCERTAINTY_PERCENTILE,
    '--quality-min-host', QUALITY_MIN_HOST,
    '--quality-min-sigma-mask-fraction', QUALITY_MIN_SIGMA_MASK_FRACTION,
    '--quality-min-valid-fraction', QUALITY_MIN_VALID_FRACTION,
    '--quality-min-resolution-ratio', QUALITY_MIN_RESOLUTION_RATIO,
    '--quality-high-conf-min-resolution-ratio', QUALITY_HIGH_CONF_MIN_RESOLUTION_RATIO,
    '--quality-max-fit-recon-mae', QUALITY_MAX_FIT_RECON_MAE,
    '--quality-max-sigma-recon-mae', QUALITY_MAX_SIGMA_RECON_MAE,
    '--quality-max-sigma-residual-to-feature', QUALITY_MAX_SIGMA_RESIDUAL_TO_FEATURE,
    '--quality-min-host-sigma-fraction', QUALITY_MIN_HOST_SIGMA_FRACTION,
    '--quality-min-absorption-depth', QUALITY_MIN_ABSORPTION_DEPTH,
    '--real-sdss-systematic-frac', REAL_SDSS_SYSTEMATIC_FRAC,
    '--real-sdss-lowres-extra-frac', REAL_SDSS_LOWRES_EXTRA_FRAC,
    '--quality-risk-calibration-max-samples', QUALITY_RISK_CALIBRATION_MAX_SAMPLES,
    '--quality-risk-neighbors', QUALITY_RISK_NEIGHBORS,
    '--quality-risk-gold-max-expected-error', QUALITY_RISK_GOLD_MAX_EXPECTED_ERROR,
    '--quality-risk-silver-max-expected-error', QUALITY_RISK_SILVER_MAX_EXPECTED_ERROR,
    '--science-sigma-bin-edges', SCIENCE_SIGMA_BIN_EDGES,
]

run_command(eval_cmd)


## 7. Representative Mock Spectra for Figures

Select representative mock spectra across velocity dispersion, host fraction, stellar population, metallicity, S/N, and Fe II strength. Figures are saved as PDF files.

In [ ]:
mock_example_cmd = [
    sys.executable,
    SCRIPTS_DIR / 'plot_mock_examples.py',
    '--h5', H5_PATH,
    '--output-dir', MOCK_EXAMPLE_DIR,
    '--prefix', RUN_TAG,
]

run_command(mock_example_cmd)

## 8. Read Metrics and Apply Basic Gates

Summarize the mock metrics, compare against a constant-mean baseline, and check the calibrated uncertainty coverage. These numerical gates do not replace inspection of the residual and reconstruction figures.

In [ ]:
with open(EVAL_JSON, 'r', encoding='utf-8') as handle:
    metrics = json.load(handle)

print(json.dumps(metrics, indent=2))

raw_rmse = metrics['rmse_kms']
baseline_rmse = metrics['baseline_mean_rmse_kms']
corr = metrics['sigma_corr']
coverage_1 = metrics.get('coverage_1sigma')
coverage_2 = metrics.get('coverage_2sigma')

print('\nBasic gates:')
print(f'raw RMSE / baseline = {raw_rmse:.1f} / {baseline_rmse:.1f} km/s')
print(f'sigma corr = {corr:.3f}')
if coverage_1 is not None and coverage_2 is not None:
    print(f'uncertainty coverage: 1sigma={coverage_1:.3f}, 2sigma={coverage_2:.3f}')

if raw_rmse < baseline_rmse and corr > 0.55:
    print('PASS: raw sigma recovery beats the mean baseline and has useful rank correlation.')
elif metrics.get('calibrated_rmse_kms', raw_rmse) < baseline_rmse and corr > 0.45:
    print('MARGINAL: calibrated sigma recovery beats baseline, but raw calibration still needs work.')
else:
    print('NOT READY: use diagnostics to decide whether to increase data/epochs or adjust sigma-specific model/loss.')

## 9. Greene & Ho 2006 External Validation

Run the trained model on the real SDSS spectra from Greene & Ho (2006). This is not a training/test metric; it is a real-observation sanity check with raw central sigma, validation-calibrated error bars, SDSS WDISP-based instrumental resolution, and high-confidence flags.


In [ ]:
from IPython.display import display
import numpy as np
import pandas as pd

greene_literature_csv = GREENE_LITERATURE_SIGMA_CSV if GREENE_LITERATURE_SIGMA_CSV.exists() else GREENE_LITERATURE_TEMPLATE_CSV
print('Greene manifest:', GREENE_MANIFEST_CSV)
print('Greene literature sigma table:', greene_literature_csv)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model checkpoint does not exist yet: {MODEL_PATH}')
if not H5_PATH.exists():
    raise FileNotFoundError(f'Mock HDF5 does not exist yet: {H5_PATH}')
if not GREENE_MANIFEST_CSV.exists():
    raise FileNotFoundError(f'Greene manifest does not exist yet: {GREENE_MANIFEST_CSV}')

greene_cmd = [
    sys.executable,
    SCRIPTS_DIR / 'infer_sdss_quasar_sigma.py',
    '--h5', H5_PATH,
    '--model', MODEL_PATH,
    '--manifest-csv', GREENE_MANIFEST_CSV,
    '--literature-csv', greene_literature_csv,
    '--central-calibration', 'raw',
    '--output-csv', GREENE_EXTERNAL_CSV,
    '--output-pdf', GREENE_EXTERNAL_PDF,
    '--quality-min-host', QUALITY_MIN_HOST,
    '--quality-min-sigma-mask-fraction', QUALITY_MIN_SIGMA_MASK_FRACTION,
    '--quality-min-valid-fraction', QUALITY_MIN_VALID_FRACTION,
    '--quality-min-resolution-ratio', QUALITY_MIN_RESOLUTION_RATIO,
    '--quality-high-conf-min-resolution-ratio', QUALITY_HIGH_CONF_MIN_RESOLUTION_RATIO,
    '--quality-max-fit-recon-mae', QUALITY_MAX_FIT_RECON_MAE,
    '--quality-max-sigma-recon-mae', QUALITY_MAX_SIGMA_RECON_MAE,
    '--quality-max-sigma-residual-to-feature', QUALITY_MAX_SIGMA_RESIDUAL_TO_FEATURE,
    '--quality-min-host-sigma-fraction', QUALITY_MIN_HOST_SIGMA_FRACTION,
    '--quality-min-absorption-depth', QUALITY_MIN_ABSORPTION_DEPTH,
    '--systematic-frac', REAL_SDSS_SYSTEMATIC_FRAC,
    '--lowres-extra-frac', REAL_SDSS_LOWRES_EXTRA_FRAC,
    '--quality-risk-gold-max-expected-error', QUALITY_RISK_GOLD_MAX_EXPECTED_ERROR,
    '--quality-risk-silver-max-expected-error', QUALITY_RISK_SILVER_MAX_EXPECTED_ERROR,
]
if CALIBRATION_JSON.exists():
    greene_cmd.extend(['--calibration-json', CALIBRATION_JSON])
else:
    print('Warning: CALIBRATION_JSON does not exist yet; real-SDSS inference will use raw sigma/error scale.')

run_command(greene_cmd)

greene_df = pd.read_csv(GREENE_EXTERNAL_CSV)
has_lit = greene_df['sigma_lit_kms'].notna() if 'sigma_lit_kms' in greene_df else pd.Series(False, index=greene_df.index)
high_conf = greene_df['high_confidence'].astype(bool) if 'high_confidence' in greene_df else pd.Series(False, index=greene_df.index)

def summarize_external(label, mask, pred_col):
    subset = greene_df[mask].copy()
    if len(subset) == 0:
        return None
    delta = subset[pred_col] - subset['sigma_lit_kms']
    corr = np.corrcoef(subset['sigma_lit_kms'], subset[pred_col])[0, 1] if len(subset) > 2 else np.nan
    return {
        'subset': label,
        'prediction': pred_col,
        'n': int(len(subset)),
        'median_pred': float(np.nanmedian(subset[pred_col])),
        'median_lit': float(np.nanmedian(subset['sigma_lit_kms'])),
        'median_delta': float(np.nanmedian(delta)),
        'mean_delta': float(np.nanmean(delta)),
        'rmse_kms': float(np.sqrt(np.nanmean(delta ** 2))),
        'mae_kms': float(np.nanmean(np.abs(delta))),
        'corr': float(corr),
    }

summary_rows = []
for subset_name, subset_mask in [('all literature', has_lit), ('high confidence', has_lit & high_conf), ('low confidence', has_lit & ~high_conf)]:
    for pred_col in ['pred_sigma_raw_kms', 'pred_sigma_affine_kms', 'pred_sigma_kms']:
        if pred_col in greene_df:
            row = summarize_external(subset_name, subset_mask, pred_col)
            if row is not None:
                summary_rows.append(row)

print('\nExternal validation summary:')
print(f'N spectra = {len(greene_df)}')
print(f'N with literature sigma = {int(has_lit.sum())}')
print(f'N high confidence = {int(high_conf.sum())}')
external_summary = pd.DataFrame(summary_rows)
display(external_summary)

if CALIBRATION_JSON.exists():
    with open(CALIBRATION_JSON, 'r', encoding='utf-8') as handle:
        calibration_payload_for_print = json.load(handle)
    print(
        'Calibration warning: central-value affine calibration is '
        f"sigma_cal = {calibration_payload_for_print.get('sigma_affine_slope', 1.0):.3f} * sigma_raw "
        f"+ {calibration_payload_for_print.get('sigma_affine_intercept', 0.0):.1f}. "
        'For low-sigma Greene objects this can create an artificial floor near 180 km/s.'
    )

show_cols = [
    'sdss_name', 'science_grade', 'high_confidence', 'central_calibration',
    'pred_sigma_raw_kms', 'pred_sigma_affine_kms', 'pred_sigma_kms', 'pred_sigma_err_kms',
    'pred_sigma_cat_kms', 'pred_sigma_fe_kms', 'pred_sigma_cahk_kms',
    'region_weight_cat', 'region_weight_fe', 'region_weight_cahk',
    'risk_expected_abs_error_kms', 'risk_p68_abs_error_kms', 'risk_reliability_score',
    'sigma_lit_kms', 'sigma_lit_err_kms', 'delta_lit_kms',
    'pred_host_fraction', 'pred_host_sigma_fraction', 'host_absorption_depth_proxy',
    'fit_recon_mae', 'sigma_recon_mae', 'sigma_recon_resid_std',
    'median_snr', 'median_inst_sigma_kms', 'pred_sigma_resolution_ratio',
    'sigma_mask_fraction', 'valid_fraction', 'near_resolution_limit',
]
display(greene_df[[col for col in show_cols if col in greene_df.columns]])
print('saved CSV:', GREENE_EXTERNAL_CSV)
print('saved PDF:', GREENE_EXTERNAL_PDF)


## 10. Greene & Ho Real-SDSS Reconstruction Diagnostics

Plot each Greene & Ho spectrum against the model reconstruction and predicted host/AGN components. Use this to diagnose whether external-validation failures come from wrong line broadening, bad decomposition, poor coverage, low host fraction, or contamination near CaT/Fe/CaK windows.


In [ ]:
from IPython.display import display
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from types import SimpleNamespace
from matplotlib.backends.backend_pdf import PdfPages
from infer_sdss_quasar_sigma import (
    read_sdss_spectrum,
    build_rest_frame_channels,
    load_manifest,
    load_calibration,
    reconstruction_diagnostics,
    build_science_quality_report,
    SIGMA_SCALE_KMS as INFER_SIGMA_SCALE_KMS,
)
from train_hybrid_sdss import load_h5_assets, load_h5_physical_config, velocity_pixel_scale
from physmodel import HybridAGNNet

AUTO_CHANNELS = {
    1: ('flux',),
    3: ('flux', 'ivar', 'mask'),
    4: ('flux', 'ivar', 'mask', 'sigma_mask'),
    5: ('flux', 'ivar', 'mask', 'sigma_flux', 'sigma_mask'),
    6: ('flux', 'ivar', 'mask', 'sigma_flux', 'inst_sigma', 'sigma_mask'),
    9: ('flux', 'ivar', 'mask', 'sigma_flux', 'inst_sigma', 'region_cat', 'region_fe', 'region_cahk', 'sigma_mask'),
    10: ('flux', 'ivar', 'mask', 'sigma_flux', 'inst_sigma', 'region_cat', 'region_fe', 'region_cahk', 'agn_log_l5100', 'sigma_mask'),
}

def finite_percentile_limits(arrays, lo=0.5, hi=99.5, pad_fraction=0.08):
    values = []
    for arr in arrays:
        if arr is None:
            continue
        arr = np.asarray(arr)
        values.append(arr[np.isfinite(arr)])
    values = np.concatenate(values) if values else np.array([0.0, 1.0])
    if values.size == 0:
        return -0.1, 1.1
    y0, y1 = np.nanpercentile(values, [lo, hi])
    if not np.isfinite(y0) or not np.isfinite(y1) or y1 <= y0:
        y0, y1 = float(np.nanmin(values)), float(np.nanmax(values))
    span = max(y1 - y0, 1e-3)
    return float(y0 - pad_fraction * span), float(y1 + pad_fraction * span)

def shade_boolean_regions(ax, wave, mask, color='tab:green', alpha=0.08):
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0 or not mask.any():
        return
    edges = np.flatnonzero(np.diff(mask.astype(int)) != 0) + 1
    starts = np.r_[0, edges]
    ends = np.r_[edges, mask.size]
    for start, end in zip(starts, ends):
        if mask[start]:
            ax.axvspan(float(wave[start]), float(wave[end - 1]), color=color, alpha=alpha, lw=0)

def load_lit_table(path):
    if not Path(path).exists():
        return {}
    table = pd.read_csv(path)
    return {row['sdss_name']: row for _, row in table.iterrows()}

wave_grid, template_library, fe_template, _, _, sigma_min, sigma_max = load_h5_assets(str(H5_PATH))
physical_config = load_h5_physical_config(str(H5_PATH))
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
state = torch.load(MODEL_PATH, map_location=device)
first_conv = state.get('backbone.0.weight')
input_channels = int(first_conv.shape[1]) if first_conv is not None else 5
greene_channels = AUTO_CHANNELS.get(input_channels)
if greene_channels is None:
    raise ValueError(f'Cannot infer channel names for checkpoint with {input_channels} input channels')

greene_model = HybridAGNNet(
    input_dim=len(wave_grid),
    template_library=template_library,
    dv_per_pixel=velocity_pixel_scale(wave_grid),
    input_channels=len(greene_channels),
    wave_grid=wave_grid,
    fe_template=fe_template,
    instrument_sigma_kms=physical_config['instrument_sigma_kms'],
    template_sigma_inst_kms=physical_config['template_sigma_inst_kms'],
    sigma_min=sigma_min,
    sigma_max=sigma_max,
).to(device)
greene_model.load_state_dict(state)
greene_model.eval()

calibration = load_calibration(str(CALIBRATION_JSON) if CALIBRATION_JSON.exists() else None)
slope = float(calibration.get('sigma_affine_slope', 1.0))
intercept = float(calibration.get('sigma_affine_intercept', 0.0))
err_scale = float(calibration.get('sigma_error_scale', 1.0))
err_floor = float(calibration.get('sigma_error_floor_kms', 0.0))
quality_cuts = calibration.get('quality_cuts', {})
real_quality_cuts = calibration.get('real_sdss_quality_cuts', {}) if isinstance(calibration, dict) else {}
quality_cuts = {**quality_cuts, **real_quality_cuts}
max_error = quality_cuts.get('max_calibrated_sigma_err_kms', np.inf)
min_host = quality_cuts.get('min_pred_host_fraction', QUALITY_MIN_HOST)
min_sigma_mask = quality_cuts.get('min_sigma_mask_fraction', QUALITY_MIN_SIGMA_MASK_FRACTION)
min_valid = quality_cuts.get('min_valid_fraction', QUALITY_MIN_VALID_FRACTION)
min_resolution_ratio = quality_cuts.get('min_sigma_resolution_ratio', QUALITY_MIN_RESOLUTION_RATIO)
advisory_resolution_ratio = quality_cuts.get('advisory_sigma_resolution_ratio', max(1.25, float(min_resolution_ratio)))
high_conf_min_resolution_ratio = quality_cuts.get('high_conf_min_sigma_resolution_ratio', advisory_resolution_ratio)
max_fit_recon_mae = quality_cuts.get('max_fit_recon_mae', QUALITY_MAX_FIT_RECON_MAE)
max_sigma_recon_mae = quality_cuts.get('max_sigma_recon_mae', QUALITY_MAX_SIGMA_RECON_MAE)
max_sigma_residual_to_feature = quality_cuts.get('max_sigma_residual_to_feature', QUALITY_MAX_SIGMA_RESIDUAL_TO_FEATURE)
min_host_sigma_fraction = quality_cuts.get('min_host_sigma_fraction', QUALITY_MIN_HOST_SIGMA_FRACTION)
min_absorption_depth = quality_cuts.get('min_absorption_depth', QUALITY_MIN_ABSORPTION_DEPTH)

manifest_rows = load_manifest(SimpleNamespace(manifest_csv=str(GREENE_MANIFEST_CSV), input_dir=None, max_spectra=None))
lit_table = load_lit_table(GREENE_LITERATURE_SIGMA_CSV)
mask_idx = greene_channels.index('mask') if 'mask' in greene_channels else None
sigma_mask_idx = greene_channels.index('sigma_mask') if 'sigma_mask' in greene_channels else None
summary_rows = []

with PdfPages(GREENE_RECON_PDF) as pdf:
    for row in manifest_rows:
        sdss_name = row.get('sdss_name') or Path(row.get('local_path', '')).stem
        path = row.get('local_path') or row.get('path') or row.get('filename')
        spectrum = read_sdss_spectrum(path)
        z_text = row.get('z', '')
        redshift = float(z_text) if z_text not in ('', None) else spectrum['redshift']
        x, meta = build_rest_frame_channels(
            spectrum['observed_wave'],
            spectrum['flux'],
            spectrum['ivar'],
            spectrum['valid'],
            spectrum.get('inst_sigma_kms'),
            float(redshift),
            wave_grid,
            greene_channels,
            fallback_inst_sigma_kms=physical_config['instrument_sigma_kms'],
            target_common_resolution_sigma_kms=physical_config.get('common_resolution_sigma', 0.0),
        )
        with torch.no_grad():
            tensor = torch.from_numpy(x).unsqueeze(0).to(device)
            pred, logvar, _, host_pred, recon, agn_part, stellar_part = greene_model(tensor)
        pred_raw = float(pred.cpu().numpy().reshape(-1)[0])
        pred_err_raw = float(np.exp(0.5 * np.clip(float(logvar.cpu().numpy().reshape(-1)[0]), -6.0, 4.0)) * INFER_SIGMA_SCALE_KMS)
        pred_cal = slope * pred_raw + intercept
        stat_err_cal = float(np.sqrt((err_scale * abs(slope) * pred_err_raw) ** 2 + err_floor**2))
        resolution_ratio = pred_raw / max(float(meta['median_inst_sigma_kms']), 1e-6)
        lowres_strength = float(np.clip((1.5 - resolution_ratio) / 0.5, 0.0, 1.0))
        systematic_floor = float(np.sqrt((REAL_SDSS_SYSTEMATIC_FRAC * pred_raw) ** 2 + (lowres_strength * REAL_SDSS_LOWRES_EXTRA_FRAC * pred_raw) ** 2))
        pred_err_cal = float(np.sqrt(stat_err_cal ** 2 + systematic_floor ** 2))
        pred_host = float(host_pred.cpu().numpy().reshape(-1)[0])

        flux = x[0]
        recon_np = recon.cpu().numpy()[0]
        host_np = stellar_part.cpu().numpy()[0]
        agn_np = agn_part.cpu().numpy()[0]
        valid = np.asarray(meta.get('valid', np.isfinite(flux)), dtype=bool) & np.isfinite(flux)
        fit_mask = (x[mask_idx] > 0.5) if mask_idx is not None else valid
        sigma_mask = (x[sigma_mask_idx] > 0.5) if sigma_mask_idx is not None else np.zeros_like(valid, dtype=bool)
        plot_mask = valid & np.isfinite(recon_np)
        residual = flux - recon_np
        fit_metric_mask = plot_mask & fit_mask
        sigma_metric_mask = plot_mask & sigma_mask
        recon_mae_fit = float(np.nanmean(np.abs(residual[fit_metric_mask]))) if fit_metric_mask.any() else np.nan
        recon_mae_sigma = float(np.nanmean(np.abs(residual[sigma_metric_mask]))) if sigma_metric_mask.any() else np.nan
        sigma_resid_std = float(np.nanstd(residual[sigma_metric_mask])) if sigma_metric_mask.any() else np.nan
        recon_diag = reconstruction_diagnostics(x, recon_np, agn_np, host_np)
        recon_mae_fit = float(recon_diag.get('fit_recon_mae', recon_mae_fit))
        recon_mae_sigma = float(recon_diag.get('sigma_recon_mae', recon_mae_sigma))
        sigma_resid_std = float(recon_diag.get('sigma_recon_resid_std', sigma_resid_std))
        quality_report = build_science_quality_report(
            pred_sigma=pred_raw,
            pred_sigma_err=pred_err_cal,
            max_error=max_error,
            pred_host=pred_host,
            min_host=float(min_host),
            sigma_mask_fraction=float(meta['sigma_mask_fraction']),
            min_sigma_mask=float(min_sigma_mask),
            valid_fraction=float(meta['valid_fraction']),
            min_valid=float(min_valid),
            resolution_ratio=resolution_ratio,
            min_resolution_ratio=float(min_resolution_ratio),
            advisory_resolution_ratio=float(advisory_resolution_ratio),
            high_conf_min_resolution_ratio=float(high_conf_min_resolution_ratio),
            fit_recon_mae=float(recon_mae_fit),
            max_fit_recon_mae=float(max_fit_recon_mae),
            sigma_recon_mae=float(recon_mae_sigma),
            max_sigma_recon_mae=float(max_sigma_recon_mae),
            sigma_residual_to_feature=float(recon_diag.get('sigma_residual_to_feature', np.inf)),
            max_sigma_residual_to_feature=float(max_sigma_residual_to_feature),
            host_sigma_fraction=float(recon_diag.get('pred_host_sigma_fraction', 0.0)),
            min_host_sigma_fraction=float(min_host_sigma_fraction),
            absorption_depth=float(recon_diag.get('host_absorption_depth_proxy', 0.0)),
            min_absorption_depth=float(min_absorption_depth),
            sigma_min=float(sigma_min),
            sigma_max=float(sigma_max),
        )
        high_conf = bool(quality_report['high_confidence'])
        science_grade = str(quality_report['science_grade'])

        lit = lit_table.get(sdss_name)
        lit_text = 'lit sigma: ...'
        lit_sigma = np.nan
        lit_err = np.nan
        if lit is not None and pd.notna(lit.get('sigma_lit_kms', np.nan)):
            lit_sigma = float(lit['sigma_lit_kms'])
            lit_err = float(lit['sigma_lit_err_kms']) if pd.notna(lit.get('sigma_lit_err_kms', np.nan)) else np.nan
            lit_text = f'lit sigma={lit_sigma:.1f} +/- {lit_err:.1f} km/s'

        summary_rows.append({
            'sdss_name': sdss_name,
            'local_path': path,
            'redshift': float(redshift),
            'sigma_lit_kms': lit_sigma,
            'sigma_lit_err_kms': lit_err,
            'pred_sigma_raw_kms': pred_raw,
            'pred_sigma_cal_kms': pred_cal,
            'pred_sigma_err_cal_kms': pred_err_cal,
            'stat_sigma_err_cal_kms': stat_err_cal,
            'systematic_sigma_floor_kms': systematic_floor,
            'median_inst_sigma_kms': float(meta['median_inst_sigma_kms']),
            'pred_sigma_resolution_ratio': resolution_ratio,
            'pred_host_fraction': pred_host,
            'median_snr': float(meta['median_snr']),
            'valid_fraction': float(meta['valid_fraction']),
            'sigma_mask_fraction': float(meta['sigma_mask_fraction']),
            'common_resolution_degraded': bool(meta.get('common_resolution_degraded', 0.0)),
            **quality_report,
            'science_grade': science_grade,
            'high_confidence': high_conf,
            'recon_mae_fit': recon_mae_fit,
            'recon_mae_sigma': recon_mae_sigma,
            'sigma_resid_std': sigma_resid_std,
            'pred_host_sigma_fraction': float(recon_diag.get('pred_host_sigma_fraction', np.nan)),
            'host_absorption_depth_proxy': float(recon_diag.get('host_absorption_depth_proxy', np.nan)),
            'flux_absorption_depth_proxy': float(recon_diag.get('flux_absorption_depth_proxy', np.nan)),
            'sigma_residual_to_feature': float(recon_diag.get('sigma_residual_to_feature', np.nan)),
        })

        fig, axes = plt.subplots(
            3,
            1,
            figsize=(11.5, 7.2),
            sharex=True,
            constrained_layout=True,
            gridspec_kw={'height_ratios': [1.25, 1.05, 0.65]},
        )
        for ax in axes:
            shade_boolean_regions(ax, wave_grid, sigma_mask, color='tab:green', alpha=0.075)

        axes[0].plot(wave_grid[plot_mask], flux[plot_mask], color='0.25', lw=0.55, alpha=0.65, label='SDSS normalized flux')
        axes[0].plot(wave_grid[plot_mask], recon_np[plot_mask], color='tab:blue', lw=1.0, label='Model reconstruction')
        axes[0].set_ylabel('Flux')
        axes[0].set_ylim(*finite_percentile_limits([flux[plot_mask], recon_np[plot_mask]], 0.2, 99.8, 0.08))
        axes[0].legend(frameon=False, fontsize=7.5, loc='upper right')

        axes[1].plot(wave_grid[plot_mask], flux[plot_mask], color='0.45', lw=0.45, alpha=0.35, label='SDSS flux')
        axes[1].plot(wave_grid[plot_mask], recon_np[plot_mask], color='tab:blue', lw=0.95, label='Reconstruction')
        axes[1].plot(wave_grid[plot_mask], host_np[plot_mask], color='tab:green', lw=1.0, ls='--', label='Predicted host')
        axes[1].plot(wave_grid[plot_mask], agn_np[plot_mask], color='tab:red', lw=0.9, ls=':', label='Predicted AGN')
        component_zoom_mask = plot_mask & (fit_mask | sigma_mask)
        if not component_zoom_mask.any():
            component_zoom_mask = plot_mask
        axes[1].set_ylim(*finite_percentile_limits([flux[component_zoom_mask], recon_np[component_zoom_mask], host_np[component_zoom_mask], agn_np[component_zoom_mask]], 1.0, 99.0, 0.12))
        axes[1].set_ylabel('Component zoom')
        axes[1].legend(frameon=False, fontsize=7.5, loc='upper right', ncol=4)

        axes[2].axhline(0.0, color='0.25', lw=0.8)
        axes[2].plot(wave_grid[plot_mask], residual[plot_mask], color='0.15', lw=0.45, alpha=0.65, label='Flux - reconstruction')
        axes[2].set_ylim(*finite_percentile_limits([residual[fit_metric_mask] if fit_metric_mask.any() else residual[plot_mask]], 1.0, 99.0, 0.15))
        axes[2].set_ylabel('Residual')
        axes[2].set_xlabel('Rest wavelength [Angstrom]')

        axes[0].set_title(
            f'{sdss_name}  z={float(redshift):.4f}  raw sigma={pred_raw:.1f}, cal sigma={pred_cal:.1f} +/- {pred_err_cal:.1f} km/s; '
            f'{lit_text}; host={pred_host:.2f}; S/N={float(meta["median_snr"]):.1f}; grade={science_grade}; high_conf={high_conf}',
            loc='left',
            fontsize=8.5,
        )
        pdf.savefig(fig)
        plt.close(fig)

recon_df = pd.DataFrame(summary_rows)
recon_df.to_csv(GREENE_RECON_CSV, index=False)
print('saved reconstruction PDF:', GREENE_RECON_PDF)
print('saved reconstruction CSV:', GREENE_RECON_CSV)
display(recon_df[[
    'sdss_name', 'science_grade', 'high_confidence', 'sigma_lit_kms',
    'pred_sigma_raw_kms', 'pred_sigma_cal_kms', 'pred_sigma_err_cal_kms',
    'pred_host_fraction', 'pred_host_sigma_fraction', 'host_absorption_depth_proxy',
    'median_snr', 'sigma_mask_fraction', 'common_resolution_degraded',
    'recon_mae_fit', 'recon_mae_sigma', 'sigma_resid_std'
]])


## 11. APJ External-Validation Figure

Regenerate the publication-style Greene & Ho comparison figure from the external-validation CSV.


In [ ]:
# Publication-quality Greene & Ho (2006) comparison figure for ApJ-style output.
# The plotting logic lives in scripts/plot_greene_apj_comparison.py to avoid
# notebook-buffer drift and to keep the final figure reproducible.

APJ_GREENE_COMPARISON_PDF = FIGURE_DIR / f'{RUN_TAG}_greene2006_external_validation_apj.pdf'
plot_apj_cmd = [
    sys.executable,
    SCRIPTS_DIR / 'plot_greene_apj_comparison.py',
    '--run-tag', RUN_TAG,
    '--input-csv', GREENE_EXTERNAL_CSV,
    '--output-pdf', APJ_GREENE_COMPARISON_PDF,
]
run_command(plot_apj_cmd, force=True)
print('saved:', APJ_GREENE_COMPARISON_PDF)
